Copyright &copy; 2024 Scott Jensen, San Jose State University

<a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">This notebook</span> by <span xmlns:cc="http://creativecommons.org/ns#" property="cc:attributionName">Scott Jensen,Ph.D.</span> is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/">Creative Commons Attribution-ShareAlike 4.0 International License</a>.

#Summarization

This notebook employs Google Gemini and LangChain workflows on Databricks to process and summarize customer reviews from the restaurant_reviews_for_summarization_table. The primary objective is to analyze reviews for a specific set of restaurant business IDs and generate succinct, actionable summaries. These summaries highlight key insights, including sentiment analysis, thematic trends, and performance across categories such as food quality, service, and pricing.

In [0]:
pip install -Uq google-generativeai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

dbutils.library.restartPython()



In [0]:
pip install -q langchain

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
pip install -q langchain-google-genai

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

#Sentiment Positive or Negative
We are defining a function to receive data for a restaurant. Our sentiment will be curated by positive or negative reviews. Our parameter for positive reviews is greater than 2 stars, and for negative reviews, it will be less than 2 stars.

In [0]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

model = ChatGoogleGenerativeAI(model="gemini-1.5-flash")

In [0]:
spark.sql("DESCRIBE restaurant_reviews_for_summarization_table").show()


+-----------+--------------------+-------+
|   col_name|           data_type|comment|
+-----------+--------------------+-------+
|    user_id|              string|   NULL|
|business_id|              string|   NULL|
|     useful|              bigint|   NULL|
|  review_id|              string|   NULL|
|      hours|struct<Friday:str...|   NULL|
|       city|              string|   NULL|
|postal_code|              string|   NULL|
|       text|              string|   NULL|
|       name|              string|   NULL|
|      stars|              double|   NULL|
+-----------+--------------------+-------+



In [0]:
def get_positive_negative_reviews(bus_id_sentiment_list):
  bus_id = bus_id_sentiment_list[0]
  sentiment = bus_id_sentiment_list[1]
  if spark.catalog._jcatalog.tableExists('restaurant_reviews_for_summarization_table'):
    print('restaurant_reviews_for_summarization_table is already loaded in memory')
  else:
    spark.sql(f"""
      CREATE TABLE restaurant_reviews_for_summarization_table
      USING PARQUET 
      LOCATION '/user/hive/warehouse/restaurant_reviews_for_summarization_table' 
    """)
    print('restaurant_reviews_for_summarization_table rebuilt from parquet files.')

  if(sentiment == 'positive'):
    df_positive_row = spark.sql(f"""
     SELECT text, stars 
     From restaurant_reviews_for_summarization_table
     WHERE stars >= 3 and business_id = '{bus_id}'
    """).collect()

    positive_list = [ ]
    for row in df_positive_row: 
      positive_list.append(row.text)

    return {'sentiment': 'positive',
            'review_list': positive_list    
    }
    
  if(sentiment == 'negative'):
    df_negative_row = spark.sql(f"""
     SELECT text, stars 
     From restaurant_reviews_for_summarization_table
     WHERE stars < 3 and business_id = '{bus_id}'
    """).collect()

    negative_list = [ ]
    for row in df_negative_row: 
      negative_list.append(row.text)

    return {'sentiment': 'negative',
            'review_list': negative_list    
    }

In [0]:
spark.sql("REFRESH TABLE restaurant_reviews_for_summarization_table")

DataFrame[]

In [0]:
spark.sql("""
    SELECT business_id, stars, text
    FROM restaurant_reviews_for_summarization_table
    WHERE stars < 3 AND business_id = '196CWwMAtAcA21jYiMyRzg'
""").show()


+-----------+-----+----+
|business_id|stars|text|
+-----------+-----+----+
+-----------+-----+----+



In [0]:
print(get_positive_negative_reviews(['196CWwMAtAcA21jYiMyRzg', 'positive']))

restaurant_reviews_for_summarization_table is already loaded in memory
{'sentiment': 'positive', 'review_list': ['I really, really enjoyed taking a break from walking around to enjoy a delicious meal here. I ordered a bowl of Cajun Gumbo and it was simply fantastic. Well seasoned and delicious to the last bite. My date ordered a shrimp Po boy that looked quite tasty as well. Very fluffy bread that must have been good since he ate every bite. We walked away extremely full and satisfied', 'Over priced bad food. Worst meal we had in NOLA. Bad service also. The restaurant looks worn. Bathroom is scary. Only good thing about it is the location.', 'Yum!  I had the trio tonight!  Shrimp gumbo, shrimp etouffe and jambalaya!  Hit the spot. Very tasty', 'The Voodoo Punch is amazing! Four different flavors of Malibu rum topped with different fruit juice with a fresh squeeze of orange on top makes this a perfect warm weather refresher! And at only $10, is sure to get you buzzed!', 'Was there about

In [0]:
print(get_positive_negative_reviews(['196CWwMAtAcA21jYiMyRzg', 'negative']))

restaurant_reviews_for_summarization_table is already loaded in memory
{'sentiment': 'negative', 'review_list': []}


#Prompt
The template we employ will prompt our LLM to generate a summary that aligns with our desired objectives. We leverage LangChain, which is processed through our template, to provide variables. By acquiring this capability, we can optimize our summarization for future customers who may be seeking specific attributes or aspects of a restaurant. 





In [0]:
from langchain_core.prompts import PromptTemplate

format_instructions = """
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{"sentiment": string,
"categories": [{"category": string, "themes": [{"theme": string, "description": "string", "reviews": integer}]}]}
```
"""

template = """You are a data analyst summarizing the main themes in a list of {sentiment} reviews for a single restaurant. First, identify the main themes of {sentiment} aspects of the restaurant that are discussed in multiple reviews. Next, summarize each theme as a description using a detailed multiple paragraph format. Next, group common themes into broader categories. Then, as output provide the sentiment as to whether the reviews are negative or positive, a bullet point list of the categories, the theme descriptions within each category in a detailed paragraph format, and a count of the number of reviews that were {sentiment} about that theme."

{format_instructions}

% USER INPUT:
{review_list}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(
    input_variables=["sentiment", "review_list"],
    partial_variables={"format_instructions": format_instructions},
    template=template,
)



#Getting the Positive Reviews
Here is the runnable chain that displays only positive reviews. This feature is beneficial for customers who simply wish to assess the strengths of a restaurant. Customers can discern whether the food quality was satisfactory or if the staff provided exceptional service.

In [0]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.base import RunnableEach
import json
import pprint

runnable_restaurants = RunnableLambda(get_positive_negative_reviews)

parser = JsonOutputParser()

chain = runnable_restaurants | prompt_template | model | parser

output = chain.invoke(['196CWwMAtAcA21jYiMyRzg', 'positive'])
print(json.dumps(output, indent=2))

restaurant_reviews_for_summarization_table is already loaded in memory
{
  "sentiment": "mixed",
  "categories": [
    {
      "category": "Food Quality",
      "themes": [
        {
          "theme": "Delicious Cajun/Creole Dishes",
          "description": "Many reviewers raved about the deliciousness of specific Cajun and Creole dishes.  The Cajun combo, featuring gumbo, jambalaya, and crawfish pie, was frequently praised for its flavorful and well-seasoned components.  Other dishes like the blackened alligator, shrimp \u00e9touff\u00e9e, and ragin' Cajun pasta also received high marks for their authentic taste and generous portions. The crawfish pie, often described as a savory empanada, stood out for its unique and scrumptious flavor.  However, some experiences varied, with a few mentioning dishes that lacked spice or were bland.  This suggests that while the restaurant excels in certain dishes, consistency may be an issue.",
          "reviews": 67
        },
        {
         

#Output as Markdown
We’re integrating LangChain to automate the analysis and summarization of customer reviews into easily understandable formats. This approach transforms raw JSON data into actionable insights, such as sentiment, categories, and themes, simplifying the interpretation of feedback related to food quality, service, and pricing.

By implementing this approach, we improve usability, streamline decision-making, and demonstrate the efficiency of LangChain in processing real-world data. This ensures that stakeholders can quickly grasp the key takeaways from substantial datasets.


In [0]:
from IPython.display import display as python_display
from IPython.display import Markdown

output_list = []
line_break = '<br/>'

title = f"#{output['sentiment'].title()} Reviews"
output_list.append(title)

for category in output['categories']:
    output_list.append(f"##Category: {category['category'].title()}")
    
    for theme in category['themes']:
        output_list.append(f"* **{theme['theme'].title()}:**{line_break}{theme['description']}")

python_display(Markdown('\n'.join(output_list)))

#Mixed Reviews
##Category: Food Quality
* **Delicious Cajun/Creole Dishes:**<br/>Many reviewers raved about the deliciousness of specific Cajun and Creole dishes.  The Cajun combo, featuring gumbo, jambalaya, and crawfish pie, was frequently praised for its flavorful and well-seasoned components.  Other dishes like the blackened alligator, shrimp étouffée, and ragin' Cajun pasta also received high marks for their authentic taste and generous portions. The crawfish pie, often described as a savory empanada, stood out for its unique and scrumptious flavor.  However, some experiences varied, with a few mentioning dishes that lacked spice or were bland.  This suggests that while the restaurant excels in certain dishes, consistency may be an issue.
* **Subpar Dishes And Inconsistent Quality:**<br/>A significant number of reviews criticized the inconsistent quality of the food.  While some dishes received glowing reviews, others were described as bland, flavorless, cold, or even inedible.  The jambalaya, in particular, was a frequent target of negative feedback, often described as dry, mushy, or tasting like a boxed mix.  Other dishes like the po'boys, crab cakes, and fried seafood were also occasionally criticized for being overcooked, under-seasoned, or containing uncleaned shrimp. These inconsistencies point to a potential problem with food preparation and quality control.
* **Overpriced Portions:**<br/>Several reviews mentioned that the portion sizes were small relative to the price.  While some dishes were generously portioned, others were considered too small, especially considering the restaurant's location in a touristy area.  This perception of poor value for money contributed to some negative experiences, particularly for those who expected larger portions given the cost.
##Category: Service
* **Excellent And Attentive Service:**<br/>Many reviewers praised the service, highlighting the attentiveness and friendliness of the waitstaff.  Servers were often described as knowledgeable, helpful, and efficient, even during busy periods.  Some reviewers specifically mentioned servers by name, emphasizing their positive impact on the dining experience.  These positive experiences suggest that the restaurant employs capable and friendly staff who strive to provide excellent customer service.
* **Poor And Inattentive Service:**<br/>Conversely, a considerable number of reviews detailed negative experiences with the service.  Common complaints included slow service, inattentiveness, rudeness, and a lack of responsiveness to customer needs.  Some reviewers felt ignored or rushed, while others reported encountering unfriendly or even hostile staff members.  These negative experiences suggest that service quality is inconsistent and needs improvement.
##Category: Ambiance And Location
* **Great Location And Atmosphere:**<br/>The restaurant's prime location in Jackson Square, with its open-air seating and views of the square and St. Louis Cathedral, was a significant positive factor for many reviewers.  The open-air design, allowing for natural light and sounds from the square, created a lively and enjoyable atmosphere.  The historic building itself was also frequently complimented, adding to the overall charm and appeal.  Many reviewers appreciated the opportunity for people-watching and enjoying the ambiance of the French Quarter.
* **Cleanliness Concerns:**<br/>Several reviews expressed concerns about the cleanliness of the restaurant.  Some mentioned dirty tables, floors, or restrooms, suggesting a need for improved hygiene practices.  The open-air design, while appreciated by many, was also cited as a potential contributor to cleanliness issues, particularly concerning insects.  These negative comments highlight the need for consistent attention to cleanliness and hygiene.

In [0]:
def get_reviews_by_id(business_id):
    
    table_name = "restaurant_reviews_for_summarization_table"
    
    if table_name not in spark.catalog.listTables():
        
        load_table()  

    
    reviews_df = spark.sql(f"SELECT * FROM {table_name} WHERE business_id = '{business_id}'")
    return reviews_df.collect()  


In [0]:
def get_pos_neg_reviews(bus_id):
  if spark.catalog._jcatalog.tableExists('restaurant_reviews_for_summarization_table'):
    print('restaurant_reviews_for_summarization_table is already loaded in memory')
  else:
    spark.sql(f"""
      CREATE TABLE restaurant_reviews_for_summarization_table
      USING PARQUET 
      LOCATION '/user/hive/warehouse/restaurant_reviews_for_summarization_table' 
    """)
    print('restaurant_reviews_for_summarization_table rebuilt from parquet files.')

  
  df_positive_row = spark.sql(f"""
    SELECT text, stars 
    From restaurant_reviews_for_summarization_table
    WHERE stars >= 3 and business_id = '{bus_id}'
  """).collect()

  positive_list = [ ]
  for row in df_positive_row: 
     positive_list.append(row.text)

  df_negative_row = spark.sql(f"""
    SELECT text, stars
    From restaurant_reviews_for_summarization_table
    WHERE stars < 3 and business_id = '{bus_id}'
  """).collect()

  negative_list = [ ]
  for row in df_negative_row: 
    negative_list.append(row.text)
  
  positive_dictionary = {'sentiment': 'positive',
            'review_list': positive_list    
    }

  negative_dictionary = {'sentiment': 'negative',
    'review_list': negative_list     
  } 
  return positive_dictionary, negative_dictionary
 

In [0]:
print(get_pos_neg_reviews ('196CWwMAtAcA21jYiMyRzg'))

restaurant_reviews_for_summarization_table is already loaded in memory
({'sentiment': 'positive', 'review_list': ['I really, really enjoyed taking a break from walking around to enjoy a delicious meal here. I ordered a bowl of Cajun Gumbo and it was simply fantastic. Well seasoned and delicious to the last bite. My date ordered a shrimp Po boy that looked quite tasty as well. Very fluffy bread that must have been good since he ate every bite. We walked away extremely full and satisfied', 'Over priced bad food. Worst meal we had in NOLA. Bad service also. The restaurant looks worn. Bathroom is scary. Only good thing about it is the location.', 'Yum!  I had the trio tonight!  Shrimp gumbo, shrimp etouffe and jambalaya!  Hit the spot. Very tasty', 'The Voodoo Punch is amazing! Four different flavors of Malibu rum topped with different fruit juice with a fresh squeeze of orange on top makes this a perfect warm weather refresher! And at only $10, is sure to get you buzzed!', 'Was there abou

#Additional Business Summarizations

##Business ID = DVBJRvnCpkqaYl6nHroaMg

In [0]:
spark.sql("""
    SELECT business_id, stars, text
    FROM restaurant_reviews_for_summarization_table
    WHERE stars < 3 AND business_id = 'DVBJRvnCpkqaYl6nHroaMg'
""").show()

+-----------+-----+----+
|business_id|stars|text|
+-----------+-----+----+
+-----------+-----+----+



In [0]:
print(get_positive_negative_reviews(['DVBJRvnCpkqaYl6nHroaMg', 'negative']))

restaurant_reviews_for_summarization_table is already loaded in memory
{'sentiment': 'negative', 'review_list': []}


In [0]:
from langchain_core.prompts import PromptTemplate

format_instructions = """
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{"sentiment": string,
"categories": [{"category": string, "themes": [{"theme": string, "description": "string", "reviews": integer}]}]}
```
"""

template = """You are a data analyst summarizing the main themes in a list of {sentiment} reviews for a single restaurant. First, identify the main themes of {sentiment} aspects of the restaurant that are discussed in multiple reviews. Next, summarize each theme as a description using a detailed multiple paragraph format. Next, group common themes into broader categories. Then, as output provide the sentiment as to whether the reviews are negative or positive, a bullet point list of the categories, the theme descriptions within each category in a detailed paragraph format, and a count of the number of reviews that were {sentiment} about that theme."

{format_instructions}

% USER INPUT:
{review_list}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(
    input_variables=["sentiment", "review_list"],
    partial_variables={"format_instructions": format_instructions},
    template=template,
)



In [0]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.base import RunnableEach
import json
import pprint

runnable_restaurants = RunnableLambda(get_positive_negative_reviews)

parser = JsonOutputParser()

chain = runnable_restaurants | prompt_template | model | parser

output = chain.invoke(['DVBJRvnCpkqaYl6nHroaMg', 'positive'])
print(json.dumps(output, indent=2))

restaurant_reviews_for_summarization_table is already loaded in memory
{
  "sentiment": "positive",
  "categories": [
    {
      "category": "Food Quality",
      "themes": [
        {
          "theme": "Flavor",
          "description": "Multiple reviews consistently praise the exceptional and unique flavors of the food.  The descriptions often use words like \"amazing,\" \"delicious,\" \"incredible,\" and \"mouthwatering\" to describe the complex and well-balanced flavor profiles.  Reviewers frequently mention being surprised by how flavorful the vegan and vegetarian dishes are, often stating that they didn't miss the meat at all. The creative use of spices and herbs is repeatedly highlighted, with many noting the dishes' \"authentic\" Mexican taste despite the lack of meat. The freshness of the ingredients is also emphasized, contributing to the overall positive perception of flavor.",
          "reviews": 180
        },
        {
          "theme": "Freshness",
          "descrip

In [0]:
from IPython.display import display as python_display
from IPython.display import Markdown

output_list = []
line_break = '<br/>'

title = f"#{output['sentiment'].title()} Reviews"
output_list.append(title)

for category in output['categories']:
    output_list.append(f"##Category: {category['category'].title()}")
    
    for theme in category['themes']:
        output_list.append(f"* **{theme['theme'].title()}:**{line_break}{theme['description']}")

python_display(Markdown('\n'.join(output_list)))

#Positive Reviews
##Category: Food Quality
* **Flavor:**<br/>Multiple reviews consistently praise the exceptional and unique flavors of the food.  The descriptions often use words like "amazing," "delicious," "incredible," and "mouthwatering" to describe the complex and well-balanced flavor profiles.  Reviewers frequently mention being surprised by how flavorful the vegan and vegetarian dishes are, often stating that they didn't miss the meat at all. The creative use of spices and herbs is repeatedly highlighted, with many noting the dishes' "authentic" Mexican taste despite the lack of meat. The freshness of the ingredients is also emphasized, contributing to the overall positive perception of flavor.
* **Freshness:**<br/>The freshness of the ingredients is a recurring theme in the positive reviews.  Reviewers frequently mention that the food tastes fresh, using terms like "vibrant," "clean," and "locally sourced."  The daily changing menu, reflecting the availability of fresh, seasonal ingredients, is praised for its novelty and commitment to quality.  Many reviewers note that the freshness of the ingredients contributes significantly to the overall taste and quality of the dishes. The open kitchen allows customers to see the preparation process, further reinforcing the perception of freshness and care.
* **Portion Size:**<br/>Many reviews emphasize the generous portion sizes.  Reviewers often mention having leftovers, indicating that the portions are more than sufficient. The large portions are generally seen as a positive aspect, offering excellent value for the price.  The abundance of food is frequently highlighted, with many expressing surprise at the quantity served. The generous portions are often described as "galactic," "huge," and "massive," reflecting the positive perception of value and satisfaction.
* **Meat Substitutes:**<br/>The restaurant's skillful use of meat substitutes, particularly jackfruit, is a major point of praise.  Reviewers repeatedly express astonishment at how well the jackfruit mimics the taste and texture of meat, often stating that they couldn't tell the difference. The successful replication of traditional Mexican dishes using plant-based alternatives is a significant contributor to the positive feedback.  The creative and successful use of jackfruit and other substitutes is frequently cited as a reason for recommending the restaurant to both vegetarians and non-vegetarians.
##Category: Service
* **Friendliness:**<br/>The overwhelming majority of reviews praise the friendliness and helpfulness of the staff.  Reviewers frequently describe the staff as "friendly," "welcoming," "attentive," and "knowledgeable."  Many mention the staff's willingness to explain the menu and offer recommendations, creating a positive and welcoming dining experience.  The owner's personal involvement and interaction with customers are also frequently highlighted as contributing to the overall friendly atmosphere.  The staff's attentiveness and willingness to go above and beyond are frequently mentioned.
* **Speed Of Service:**<br/>Several reviews commend the speed of service, particularly considering the restaurant's popularity and frequent busyness.  Reviewers often mention receiving their food quickly after ordering, despite potential lines or crowds. This efficiency is seen as a positive aspect, enhancing the overall dining experience. The combination of fast service and high-quality food is frequently noted as a reason for recommending the restaurant.
##Category: Ambiance
* **Atmosphere:**<br/>The restaurant's atmosphere is generally described as positive and welcoming.  Reviewers use terms like "cozy," "chill," "homey," and "vibrant" to describe the ambiance.  The open kitchen, allowing customers to see the food preparation, is often mentioned as contributing to the positive atmosphere.  The casual and community-style seating arrangement is also frequently noted, along with the presence of art and decorations that add to the overall pleasant ambiance. The use of music and even live music on occasion is also a positive factor.
##Category: Value
* **Complimentary Items:**<br/>The provision of complimentary soup and coffee is a frequently mentioned positive aspect.  Reviewers appreciate this added value, enhancing the overall dining experience and contributing to the perception of generosity. The complimentary items are seen as a thoughtful touch that enhances customer satisfaction.  The inclusion of these items is often described as a "nice touch" or a "very nice touch," reflecting the positive impact on the overall experience.

##Business ID = WC8vQdCC-nSawCh2IV4epg

In [0]:
print(get_positive_negative_reviews(['WC8vQdCC-nSawCh2IV4epg', 'negative']))

restaurant_reviews_for_summarization_table is already loaded in memory
{'sentiment': 'negative', 'review_list': []}


In [0]:
from langchain_core.prompts import PromptTemplate

format_instructions = """
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{"sentiment": string,
"categories": [{"category": string, "themes": [{"theme": string, "description": "string", "reviews": integer}]}]}
```
"""

template = """You are a data analyst summarizing the main themes in a list of {sentiment} reviews for a single restaurant. First, identify the main themes of {sentiment} aspects of the restaurant that are discussed in multiple reviews. Next, summarize each theme as a description using a detailed multiple paragraph format. Next, group common themes into broader categories. Then, as output provide the sentiment as to whether the reviews are negative or positive, a bullet point list of the categories, the theme descriptions within each category in a detailed paragraph format, and a count of the number of reviews that were {sentiment} about that theme."

{format_instructions}

% USER INPUT:
{review_list}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(
    input_variables=["sentiment", "review_list"],
    partial_variables={"format_instructions": format_instructions},
    template=template,
)

In [0]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.base import RunnableEach
import json
import pprint

runnable_restaurants = RunnableLambda(get_positive_negative_reviews)

parser = JsonOutputParser()

chain = runnable_restaurants | prompt_template | model | parser

output = chain.invoke(['WC8vQdCC-nSawCh2IV4epg', 'positive'])
print(json.dumps(output, indent=2))

restaurant_reviews_for_summarization_table is already loaded in memory
{
  "sentiment": "positive",
  "categories": [
    {
      "category": "Food Quality",
      "themes": [
        {
          "theme": "Steak",
          "description": "Multiple reviews rave about the quality and preparation of the steaks.  Many describe them as \"perfectly cooked\", \"tender\", \"juicy\", and \"flavorful.\"  Specific cuts like the filet mignon and ribeye are frequently mentioned as standouts, often highlighting the use of high-quality Allen Brothers beef.  However, a few reviews mention instances of overcooked or undercooked steaks, suggesting some inconsistencies in kitchen execution, though these are exceptions rather than the rule.",
          "reviews": 30
        },
        {
          "theme": "Seafood",
          "description": "The seafood dishes also receive considerable praise.  Lobster, scallops, and sea bass are frequently cited as being fresh, expertly cooked, and delicious.  Table-sid

In [0]:
from IPython.display import display as python_display
from IPython.display import Markdown

output_list = []
line_break = '<br/>'

title = f"#{output['sentiment'].title()} Reviews"
output_list.append(title)

for category in output['categories']:
    output_list.append(f"##Category: {category['category'].title()}")
    
    for theme in category['themes']:
        output_list.append(f"* **{theme['theme'].title()}:**{line_break}{theme['description']}")

python_display(Markdown('\n'.join(output_list)))

#Positive Reviews
##Category: Food Quality
* **Steak:**<br/>Multiple reviews rave about the quality and preparation of the steaks.  Many describe them as "perfectly cooked", "tender", "juicy", and "flavorful."  Specific cuts like the filet mignon and ribeye are frequently mentioned as standouts, often highlighting the use of high-quality Allen Brothers beef.  However, a few reviews mention instances of overcooked or undercooked steaks, suggesting some inconsistencies in kitchen execution, though these are exceptions rather than the rule.
* **Seafood:**<br/>The seafood dishes also receive considerable praise.  Lobster, scallops, and sea bass are frequently cited as being fresh, expertly cooked, and delicious.  Table-side preparations like shrimp scampi add to the overall dining experience. However, a few reviews mention instances of overcooked lobster or less-than-fresh seafood, indicating some inconsistencies in quality and preparation.
* **Sides:**<br/>Many reviews highlight the deliciousness of the side dishes, with truffle mac and cheese, mashed potatoes, and asparagus being frequently mentioned as favorites.  The creamed spinach is also praised, though some reviewers note it could be improved.  However, some reviews mention instances of overcooked or under-seasoned sides.
* **Desserts:**<br/>The desserts are a significant highlight for many patrons.  Table-side preparations like Bananas Foster and Baked Alaska are frequently praised for their presentation and taste.  Other desserts like ice cream (made with liquid nitrogen), cheesecake, and coffee cake are also mentioned favorably. The complimentary desserts offered for special occasions or as a parting gift are especially appreciated. However, a few reviews mention some desserts being subpar.
* **Appetizers:**<br/>The appetizers are generally well-received, with many positive comments about the crab cakes, escargot, and pork belly lollipops.  The seafood tower is also mentioned, though some reviewers found it to be overpriced or lacking in flavor.  The quality and presentation of appetizers are praised by many.
##Category: Service
* **Attentive And Friendly Staff:**<br/>Exceptional service is a recurring theme in the positive reviews.  Many reviewers praise the attentiveness, friendliness, and professionalism of the waitstaff.  Specific servers are frequently named and lauded for their exceptional service, going above and beyond to ensure a positive dining experience.  The staff's ability to anticipate needs, promptly refill drinks, and provide helpful recommendations is consistently highlighted.  The staff's knowledge of the menu and wine list is also appreciated.
* **Table-Side Service:**<br/>The table-side preparation of certain dishes, such as Caesar salad, Steak Diane, Bananas Foster, and shrimp scampi, is a major draw for many diners.  The theatrical element of these preparations enhances the dining experience, and reviewers consistently praise both the presentation and the taste of these dishes.
* **Special Occasions:**<br/>The restaurant excels at catering to special occasions.  Many reviewers mention celebrating birthdays, anniversaries, or other events at the restaurant, noting that the staff goes out of their way to make these occasions memorable.  Complimentary desserts, personalized touches, and attentive service all contribute to a special dining experience.
##Category: Ambiance
* **Elegant And Luxurious Atmosphere:**<br/>The restaurant's elegant and luxurious ambiance is frequently praised.  Reviewers describe the decor as beautiful, modern, and sophisticated, often mentioning the unique lighting, comfortable seating, and the large fish tank as highlights.  The atmosphere is described as romantic, intimate, and perfect for special occasions.  However, some reviewers note that the restaurant can be noisy or lack privacy due to close table spacing.

##Business ID = foh6hwQxjCs0SeLT5MO1SQ

In [0]:
print(get_positive_negative_reviews(['foh6hwQxjCs0SeLT5MO1SQ', 'negative']))

restaurant_reviews_for_summarization_table is already loaded in memory
{'sentiment': 'negative', 'review_list': []}


In [0]:
from langchain_core.prompts import PromptTemplate

format_instructions = """
The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{"sentiment": string,
"categories": [{"category": string, "themes": [{"theme": string, "description": "string", "reviews": integer}]}]}
```
"""

template = """You are a data analyst summarizing the main themes in a list of {sentiment} reviews for a single restaurant. First, identify the main themes of {sentiment} aspects of the restaurant that are discussed in multiple reviews. Next, summarize each theme as a description using a detailed multiple paragraph format. Next, group common themes into broader categories. Then, as output provide the sentiment as to whether the reviews are negative or positive, a bullet point list of the categories, the theme descriptions within each category in a detailed paragraph format, and a count of the number of reviews that were {sentiment} about that theme."

{format_instructions}

% USER INPUT:
{review_list}

YOUR RESPONSE:
"""
prompt_template = PromptTemplate(
    input_variables=["sentiment", "review_list"],
    partial_variables={"format_instructions": format_instructions},
    template=template,
)

In [0]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.base import RunnableEach
import json
import pprint

runnable_restaurants = RunnableLambda(get_positive_negative_reviews)

parser = JsonOutputParser()

chain = runnable_restaurants | prompt_template | model | parser

output = chain.invoke(['foh6hwQxjCs0SeLT5MO1SQ', 'positive'])
print(json.dumps(output, indent=2))

restaurant_reviews_for_summarization_table is already loaded in memory
{
  "sentiment": "positive",
  "categories": [
    {
      "category": "Location & Convenience",
      "themes": [
        {
          "theme": "Proximity to Airport",
          "description": "Many reviewers highlighted the Wawa's convenient location near Philadelphia International Airport (PHL). This is especially beneficial for travelers returning rental cars, as it allows for a quick gas fill-up before dropping off the vehicle, avoiding potential extra charges.  The ease of access from the airport and nearby highways was repeatedly praised, making it a convenient pit-stop for travelers.  The proximity to hotels also made it a popular choice for hotel guests seeking nearby food and amenities.",
          "reviews": 18
        },
        {
          "theme": "Easy Access and Ample Parking",
          "description": "While some reviews mentioned parking lot congestion during peak hours, many others praised the ampl

In [0]:
from IPython.display import display as python_display
from IPython.display import Markdown

output_list = []
line_break = '<br/>'

title = f"#{output['sentiment'].title()} Reviews"
output_list.append(title)

for category in output['categories']:
    output_list.append(f"##Category: {category['category'].title()}")
    
    for theme in category['themes']:
        output_list.append(f"* **{theme['theme'].title()}:**{line_break}{theme['description']}")


python_display(Markdown('\n'.join(output_list)))

#Positive Reviews
##Category: Location & Convenience
* **Proximity To Airport:**<br/>Many reviewers highlighted the Wawa's convenient location near Philadelphia International Airport (PHL). This is especially beneficial for travelers returning rental cars, as it allows for a quick gas fill-up before dropping off the vehicle, avoiding potential extra charges.  The ease of access from the airport and nearby highways was repeatedly praised, making it a convenient pit-stop for travelers.  The proximity to hotels also made it a popular choice for hotel guests seeking nearby food and amenities.
* **Easy Access And Ample Parking:**<br/>While some reviews mentioned parking lot congestion during peak hours, many others praised the ample parking available at this Wawa location.  The ease of access to gas pumps and the overall layout of the parking lot were considered convenient, even with the high volume of traffic.  Reviewers appreciated the ability to quickly fill up their gas tanks without significant delays.
##Category: Food & Beverage
* **High-Quality Sandwiches:**<br/>A recurring positive theme centered on the quality and taste of the Wawa sandwiches. Reviewers frequently described the sandwiches as 'great,' 'delicious,' and 'fresh.' The ability to customize orders via a kiosk was lauded as a convenient and efficient system, ensuring that each sandwich is made exactly to the customer's preferences.  The speed of sandwich preparation, even during busy periods, was also frequently mentioned positively.  Specific sandwich types, such as the Italian hoagie and the classic cheesesteak, received high praise.
* **Wide Selection & Freshness:**<br/>Beyond sandwiches, the Wawa's diverse selection of food and beverages was a major point of praise.  Reviewers appreciated the availability of fresh fruit, pretzels, Tastycakes, and other snacks.  The overall freshness of the food items was repeatedly emphasized, contributing to the positive experience.  The availability of hot breakfast sandwiches, even at early hours, was also highlighted.
* **Convenient Ordering System:**<br/>The electronic ordering system for hoagies and sandwiches was a significant positive aspect for many reviewers.  The kiosk ordering process was seen as efficient and time-saving, especially during peak hours when lines might be long.  The ability to customize orders through the kiosk and then pick them up quickly was highly praised for its convenience.
##Category: Cleanliness & Atmosphere
* **Cleanliness Of Store & Restrooms:**<br/>The cleanliness of the store and restrooms was frequently mentioned positively.  Reviewers consistently described the Wawa as 'clean' and 'well-maintained,' suggesting a positive overall impression of the establishment's hygiene standards.  However, there were also some negative comments regarding cleanliness, particularly concerning the coffee area and overall store cleanliness.
* **Unique & Fun Atmosphere:**<br/>The unique and fun atmosphere created by playing pop music in the bathroom was a surprising but appreciated detail mentioned by several reviewers. This quirky feature contributed to a positive and memorable experience for some customers. This element is more of a niche positive, as not all reviews mentioned it.
##Category: Service
* **Friendly And Efficient Staff:**<br/>Many reviewers praised the friendliness and efficiency of the Wawa staff.  Positive comments specifically mentioned cashiers and sandwich-makers who were helpful, courteous, and provided quick service.  However, it's important to note that there were also several negative reviews mentioning rude or unhelpful staff, indicating some inconsistency in customer service.
##Category: Gas Prices
* **Competitive Gas Prices:**<br/>Several reviewers noted that the gas prices at this Wawa were competitive, especially considering its proximity to the airport.  The prices were often described as 'reasonable' or 'not inflated,' which is a significant positive for customers concerned about potential price gouging near the airport.